# RankLab full KuaiRand-Pure pipeline

This notebook is the resumable Kaggle GPU entrypoint for the full specification. It validates the official source archive or extracted hierarchy, restores compatible derived artifacts, runs the stage-checkpointed pipeline, and emits a fresh cache archive. KuaiRand randomized rows are evaluation/domain-classifier data only and are never recommender-label training data.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess

REPO_URL = 'https://github.com/kushc2004/rank-lab.git'
WORKDIR = Path('/kaggle/working/rank-lab')
try:
    if not (WORKDIR / '.git').is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORKDIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(WORKDIR), 'pull', '--ff-only'], check=True)
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        'Could not clone the public RankLab repository. Enable Internet in the Kaggle notebook settings, then Push & Run again.'
    ) from error
os.chdir(WORKDIR)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[full]'], check=True)
print('Repository:', WORKDIR)

In [ ]:
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 7:
    DEVICE = 'cuda'
    print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))
else:
    DEVICE = 'cpu'
    reason = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no CUDA device assigned'
    print('Using CPU because CUDA is unavailable or unsupported:', reason)

In [ ]:
OFFICIAL_MD5 = '0820331067a3784d9691136f772b35a7'
OFFICIAL_FILES = (
    'log_random_4_22_to_5_08_pure.csv',
    'log_standard_4_08_to_4_21_pure.csv',
    'log_standard_4_22_to_5_08_pure.csv',
    'user_features_pure.csv',
    'video_features_basic_pure.csv',
    'video_features_statistic_pure.csv',
)
# Prefer an attached official input. If it was not attached to this kernel
# version, fetch the official archive and verify its published checksum.
DOWNLOAD_IF_MISSING = True

def complete_data_directory(path):
    return path.is_dir() and all((path / name).is_file() for name in OFFICIAL_FILES)

candidates = []
for input_root in sorted(Path('/kaggle/input').glob('*')):
    for candidate in (input_root / 'KuaiRand-Pure/data', input_root / 'data', input_root):
        if complete_data_directory(candidate):
            candidates.append(candidate)
            break
archives = sorted(Path('/kaggle/input').glob('*/KuaiRand-Pure.tar.gz'))
if candidates:
    RAW_DIR = candidates[0]
elif archives:
    archive = archives[0]
    digest = hashlib.md5()
    with archive.open('rb') as source:
        for block in iter(lambda: source.read(8 * 1024 * 1024), b''):
            digest.update(block)
    if digest.hexdigest() != OFFICIAL_MD5:
        raise ValueError(f'Official archive checksum mismatch: {digest.hexdigest()}')
    raw_root = WORKDIR / 'data/raw'
    raw_root.mkdir(parents=True, exist_ok=True)
    subprocess.run(['tar', '-xzf', str(archive), '-C', str(raw_root)], check=True)
    RAW_DIR = raw_root / 'KuaiRand-Pure/data'
elif DOWNLOAD_IF_MISSING:
    subprocess.run(['bash', 'scripts/download_kuairand_pure.sh'], check=True)
    RAW_DIR = WORKDIR / 'data/raw/KuaiRand-Pure/data'
else:
    raise FileNotFoundError('Attach the official KuaiRand-Pure input or enable Internet and set DOWNLOAD_IF_MISSING=True.')
if not complete_data_directory(RAW_DIR):
    missing = [name for name in OFFICIAL_FILES if not (RAW_DIR / name).is_file()]
    raise FileNotFoundError(f'Incomplete official KuaiRand-Pure hierarchy: {missing}')
print('Raw data:', RAW_DIR)

In [ ]:
# Restore only derived data; the restore script rejects raw or unsafe paths.
artifact_archives = sorted(Path('/kaggle/input').glob('*/ranklab_artifacts.tar.gz'))
artifact_directories = [
    path for path in sorted(Path('/kaggle/input').glob('*'))
    if (path / 'outputs').is_dir() or (path / 'data/manifests').is_dir()
]
if artifact_archives:
    subprocess.run(['python', 'scripts/restore_kaggle_artifacts.py', str(artifact_archives[0])], check=True)
elif artifact_directories:
    subprocess.run(['python', 'scripts/restore_kaggle_artifacts.py', str(artifact_directories[0])], check=True)
else:
    print('No derived-artifact cache attached; the pipeline will start from its first missing stage.')

In [ ]:
# Each successful stage atomically updates outputs/full_pipeline_state.json.
command = [
    'python', 'scripts/run_full_pipeline.py',
    f'raw_dir={RAW_DIR}', f'device={DEVICE}',
]
print('$', ' '.join(map(str, command)))
subprocess.run(command, check=True)

In [ ]:
from IPython.display import Markdown, display
state_path = WORKDIR / 'outputs/full_pipeline_state.json'
print(json.dumps(json.loads(state_path.read_text()), indent=2))
report_path = WORKDIR / 'outputs/reports/full_experiment_report.md'
if report_path.is_file():
    display(Markdown(report_path.read_text()))

In [ ]:
# Persist manifests, features, models, indices, predictions, metrics, reports,
# and stage state as a notebook output. Raw KuaiRand files are never included.
subprocess.run(['python', 'scripts/publish_kaggle_artifacts.py', '--no-upload'], check=True)
print(WORKDIR / 'artifacts/kaggle/ranklab_artifacts.tar.gz')